In [1]:
# --- Cell 1: Setup and connect ---
import pandas as pd, numpy as np
from pathlib import Path
import sqlite3
import plotly.express as px

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "data-analytics"
DB = BASE / "sql" / "agriculture.db"

conn = sqlite3.connect(DB)
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

def q(sql):
    return pd.read_sql(sql, conn)

           name
0    crop_yield
1      rainfall
2  merged_yield
3  mandi_prices


In [2]:
# --- Cell 2: Q1. Top 5 crops by production in a state (Coconut excluded) ---
q("""
SELECT * FROM (
  SELECT state, crop, ROUND(SUM(production)) AS total_production,
         ROW_NUMBER() OVER (PARTITION BY state ORDER BY SUM(production) DESC) AS rnk
  FROM crop_yield WHERE crop <> 'Coconut' GROUP BY state, crop
) WHERE rnk <= 5 AND state IN ('Punjab', 'Uttar Pradesh', 'Maharashtra')
ORDER BY state, rnk""")

,state,crop,total_production,rnk
0,Maharashtra,Sugarcane,1.396764e+09,1
1,Maharashtra,Cotton(Lint),1.109723e+08,2
2,Maharashtra,Jowar,7.190475e+07,3
3,Maharashtra,Soyabean,6.352360e+07,4
4,Maharashtra,Rice,6.099203e+07,5
5,Punjab,Wheat,3.643700e+08,1
6,Punjab,Rice,2.430420e+08,2
7,Punjab,Sugarcane,1.256390e+08,3
8,Punjab,Cotton(Lint),3.463420e+07,4
9,Punjab,Maize,1.015040e+07,5


In [3]:
# --- Cell 3: Q2. Year-over-year production growth (Rice) ---
g = q("""
WITH yearly AS (SELECT year, SUM(production) AS prod FROM crop_yield WHERE crop = 'Rice' GROUP BY year)
SELECT year, ROUND(prod) AS production,
       ROUND(100.0 * (prod - LAG(prod) OVER (ORDER BY year)) / LAG(prod) OVER (ORDER BY year), 2) AS yoy_growth_pct
FROM yearly ORDER BY year""")
display(g.head())
px.bar(g, x="year", y="yoy_growth_pct", title="Rice: year-over-year production growth (%)")

,year,production,yoy_growth_pct
0,1997,61069362.0,NaN
1,1998,83064884.0,36.02
2,1999,83511141.0,0.54
3,2000,83248654.0,-0.31
4,2001,90197955.0,8.35


In [4]:
# --- Cell 4: Q3. Season-wise yield (Wheat and Rice) ---
q("""
SELECT crop, season, COUNT(*) AS records, ROUND(AVG("yield"), 2) AS avg_yield
FROM crop_yield WHERE crop IN ('Wheat', 'Rice') GROUP BY crop, season ORDER BY crop, avg_yield DESC""")

,crop,season,records,avg_yield
0,Rice,Rabi,1218,2.82
1,Rice,Summer,4088,2.45
2,Rice,Kharif,9803,2.09
3,Rice,Winter,3280,1.90
4,Rice,Autumn,3091,1.54
5,Rice,Whole Year,74,1.20
6,Wheat,Rabi,10784,2.26
7,Wheat,Whole Year,56,1.83
8,Wheat,Winter,32,1.79
9,Wheat,Summer,325,1.40


In [5]:
# --- Cell 5: Q4. Top 5 districts for Wheat yield (last 5 years) ---
q("""
SELECT state, district, ROUND(AVG("yield"), 2) AS avg_yield FROM crop_yield
WHERE crop = 'Wheat' AND year >= (SELECT MAX(year) - 4 FROM crop_yield)
GROUP BY state, district ORDER BY avg_yield DESC LIMIT 5""")

,state,district,avg_yield
0,Punjab,Sangrur,5.67
1,Punjab,Barnala,5.42
2,Punjab,Bathinda,5.40
3,Punjab,Faridkot,5.36
4,Punjab,Muktsar,5.35


In [6]:
# --- Cell 6: Q5. Most expensive commodities today (Rs/quintal) ---
p = q("""
SELECT commodity, ROUND(AVG(modal_price)) AS avg_modal_price, COUNT(*) AS markets
FROM mandi_prices GROUP BY commodity HAVING markets >= 20 ORDER BY avg_modal_price DESC LIMIT 10""")
px.bar(p, x="avg_modal_price", y="commodity", orientation="h", title="Most expensive commodities today") \
  .update_layout(yaxis={"categoryorder": "total ascending"})

In [7]:
# --- Cell 7: Close the connection ---
conn.close()